<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/ESG%20non-linear%20centered.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# standalone_nonlinear_complete.py
"""
Complete Non-Linear Effects Model with Centered Approach
Analysis of quadratic, threshold, and other non-linear specifications
"""

# ============================================================================
# SETUP AND IMPORTS
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q linearmodels pandas numpy matplotlib seaborn statsmodels

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels.panel import PanelOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import linear_reset, het_breuschpagan
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Configuration for non-linear effects analysis"""
    PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/FTSE Data'
    DATA_PATH = os.path.join(PROJECT_DIR, 'FTSE data.xlsx')
    OUTPUT_DIR = os.path.join(PROJECT_DIR, 'nonlinear_complete_outputs')

    # Consumer Staples firms (must match original)
    CONSUMER_STAPLES_FIRMS = [
        'Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG',
        'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever', 'Haleon Plc'
    ]

    # Analysis settings
    USE_CENTERED_APPROACH = True  # Set to True for centered, False for simple
    THRESHOLD_QUANTILES = [0.25, 0.5, 0.75, 0.9]
    CREATE_PLOTS = True
    SAVE_RESULTS = True

    # Output precision
    COEFF_DECIMALS = 6
    R2_DECIMALS = 6

# ============================================================================
# DATA LOADER WITH CENTERING
# ============================================================================
class DataLoaderWithCentering:
    """Load data with centered non-linear transformations"""

    def __init__(self, config):
        self.config = config
        self.df = None
        self.df_cs = None
        self.df_all = None
        self.esg_means = {}  # Store means for interpretation

    def load_and_prepare_all_transformations(self):
        """
        Load data and create ALL transformations (centered and simple)
        """
        print("Loading data and creating all non-linear transformations...")
        print("-" * 60)

        # 1. Load Excel file
        self.df = pd.read_excel(self.config.DATA_PATH)

        # 2. Set multi-index
        self.df = self.df.set_index(['Firm', 'Year']).sort_index()

        # 3. Rename Tobin's Q
        self.df = self.df.rename(columns={"Tobin's Q": "Tobin_Q"})

        # 4. Basic transformations
        self.df['Size'] = np.log(self.df['Total Assets'])
        self.df['Leverage'] = self.df['Total Liabilities'] / self.df['Total Assets']
        self.df['Tobin_Q_log'] = np.log(self.df['Tobin_Q'] + 0.001)

        # 5. Create lagged ESG variables
        print("Creating lagged ESG variables...")
        self.df['E_lag'] = self.df.groupby(level=0)['E'].shift(1)
        self.df['S_lag'] = self.df.groupby(level=0)['S'].shift(1)
        self.df['G_lag'] = self.df.groupby(level=0)['G'].shift(1)

        # 6. Create lagged controls
        self.df['ROA_lag'] = self.df.groupby(level=0)['ROA'].shift(1)
        self.df['Size_lag'] = self.df.groupby(level=0)['Size'].shift(1)
        self.df['Leverage_lag'] = self.df.groupby(level=0)['Leverage'].shift(1)

        # 7. Create sector variable
        print("Creating sector variable...")
        sectors = []
        for firm in self.df.index.get_level_values(0):
            if firm in self.config.CONSUMER_STAPLES_FIRMS:
                sectors.append('Consumer Staples')
            else:
                sectors.append('Other')

        self.df['Sector'] = sectors
        self.df['ConsumerStaples'] = (self.df['Sector'] == 'Consumer Staples').astype(int)

        # 8. CREATE NON-LINEAR TRANSFORMATIONS
        print("\nCreating non-linear transformations...")
        self.df = self._create_all_nonlinear_transformations(self.df)

        # 9. Create datasets
        self.df_cs = self.df[self.df['ConsumerStaples'] == 1].copy()
        self.df_all = self.df.copy()

        # Summary
        print("\n" + "="*60)
        print("DATA PREPARATION COMPLETE")
        print("="*60)
        print(f"Total observations: {len(self.df)}")
        print(f"Consumer Staples: {len(self.df_cs)}")
        print(f"Other sectors: {len(self.df_all) - len(self.df_cs)}")
        print(f"\nESG means (for centering):")
        for component, mean_val in self.esg_means.items():
            print(f"  {component}: {mean_val:.4f}")

        return self.df, self.df_cs, self.df_all

    def _create_all_nonlinear_transformations(self, df):
        """
        Create all non-linear transformations (centered and simple)
        """
        df = df.copy()

        # Define ESG components to transform
        esg_components = ['E_lag', 'S_lag', 'G_lag']

        print("\nA. CENTERED TRANSFORMATIONS (Recommended):")
        print("-" * 40)

        for component in esg_components:
            if component in df.columns:
                # Calculate and store mean
                mean_val = df[component].mean()
                self.esg_means[component] = mean_val

                # 1. Centered variable
                df[f'{component}_centered'] = df[component] - mean_val

                # 2. Centered squared
                df[f'{component}_sq_centered'] = df[f'{component}_centered'] ** 2

                # 3. Centered cubic
                df[f'{component}_cube_centered'] = df[f'{component}_centered'] ** 3

                print(f"  ✓ {component}_centered = {component} - {mean_val:.4f}")
                print(f"  ✓ {component}_sq_centered = ({component}_centered)²")
                print(f"  ✓ {component}_cube_centered = ({component}_centered)³")

        print("\nB. SIMPLE TRANSFORMATIONS (for comparison):")
        print("-" * 40)

        for component in esg_components:
            if component in df.columns:
                # 1. Simple squared
                df[f'{component}_sq'] = df[component] ** 2

                # 2. Simple cubic
                df[f'{component}_cube'] = df[component] ** 3

                # 3. Logarithmic (with small constant)
                min_val = df[component].min()
                constant = 1 - min_val if min_val < 1 else 0.001
                df[f'log_{component}'] = np.log(df[component] + constant)

                # 4. Square root
                shift = -min_val + 0.001 if min_val < 0 else 0
                df[f'sqrt_{component}'] = np.sqrt(df[component] + shift)

                print(f"  ✓ {component}_sq = ({component})²")
                print(f"  ✓ log_{component} = ln({component} + {constant:.3f})")

        print("\nC. THRESHOLD VARIABLES:")
        print("-" * 40)

        for component in esg_components:
            if component in df.columns:
                for q in self.config.THRESHOLD_QUANTILES:
                    try:
                        threshold = df[component].quantile(q)
                        df[f'{component}_above{q*100:.0f}'] = (df[component] > threshold).astype(int)
                        print(f"  ✓ {component}_above{q*100:.0f}: 1 if > {threshold:.4f}")
                    except:
                        pass

        print("\nD. INTERACTION TERMS (centered):")
        print("-" * 40)

        centered_components = [f'{c}_centered' for c in esg_components]
        for i in range(len(centered_components)):
            for j in range(i+1, len(centered_components)):
                var1, var2 = centered_components[i], centered_components[j]
                df[f'{var1}_{var2}_interact'] = df[var1] * df[var2]
                print(f"  ✓ {var1}_{var2}_interact = {var1} × {var2}")

        print(f"\n✓ Created {sum(['centered' in col or '_sq' in col or 'log_' in col or
                               'sqrt_' in col or '_above' in col or '_interact' in col
                               for col in df.columns])} non-linear transformations")

        return df

    def get_recommended_variables(self, centered=True):
        """
        Get recommended variable sets based on approach
        """
        if centered:
            return {
                'quadratic': ['E_lag_centered', 'E_lag_sq_centered',
                             'S_lag_centered', 'S_lag_sq_centered',
                             'G_lag_centered', 'G_lag_sq_centered'],
                'cubic': ['E_lag_centered', 'E_lag_sq_centered', 'E_lag_cube_centered',
                         'S_lag_centered', 'S_lag_sq_centered', 'S_lag_cube_centered',
                         'G_lag_centered', 'G_lag_sq_centered', 'G_lag_cube_centered'],
                'logarithmic': ['log_E_lag', 'log_S_lag', 'log_G_lag'],
                'sqrt': ['sqrt_E_lag', 'sqrt_S_lag', 'sqrt_G_lag']
            }
        else:
            return {
                'quadratic': ['E_lag', 'E_lag_sq', 'S_lag', 'S_lag_sq', 'G_lag', 'G_lag_sq'],
                'cubic': ['E_lag', 'E_lag_sq', 'E_lag_cube',
                         'S_lag', 'S_lag_sq', 'S_lag_cube',
                         'G_lag', 'G_lag_sq', 'G_lag_cube'],
                'logarithmic': ['log_E_lag', 'log_S_lag', 'log_G_lag'],
                'sqrt': ['sqrt_E_lag', 'sqrt_S_lag', 'sqrt_G_lag']
            }

# ============================================================================
# NON-LINEAR MODEL ANALYZER
# ============================================================================
class NonlinearEffectsAnalyzerComplete:
    """Complete non-linear effects analyzer with diagnostics"""

    def __init__(self, config):
        self.config = config
        self.loader = DataLoaderWithCentering(config)
        self.results = {}
        self.diagnostics = {}

    def run_complete_analysis(self, sector='cs', y_var='Tobin_Q',
                             use_centered=None, run_diagnostics=True):
        """
        Run complete non-linear analysis

        Parameters:
        -----------
        sector : str
            'cs' for Consumer Staples, 'all' for All Sectors
        y_var : str
            Dependent variable ('Tobin_Q' or 'Tobin_Q_log')
        use_centered : bool or None
            True for centered, False for simple, None for config default
        run_diagnostics : bool
            Whether to run model diagnostics
        """
        # Determine approach
        if use_centered is None:
            use_centered = self.config.USE_CENTERED_APPROACH

        approach = "CENTERED" if use_centered else "SIMPLE"

        print("\n" + "="*80)
        print(f"COMPLETE NON-LINEAR ANALYSIS - {approach} APPROACH")
        print(f"Sector: {'Consumer Staples' if sector == 'cs' else 'All Sectors'}")
        print(f"Dependent: {y_var}")
        print("="*80)

        # Load data with all transformations
        df, df_cs, df_all = self.loader.load_and_prepare_all_transformations()

        # Select dataset
        if sector == 'cs':
            analysis_df = df_cs.copy()
            sector_name = 'Consumer Staples Only'
            control_vars = ['ROA', 'Size', 'Leverage']
        else:
            analysis_df = df_all.copy()
            sector_name = 'All Sectors'
            control_vars = ['ROA', 'Size', 'Leverage', 'ConsumerStaples']

        # Get variable sets
        var_sets = self.loader.get_recommended_variables(centered=use_centered)

        # Run all model specifications
        print(f"\nRunning model specifications ({approach} approach)...")

        all_results = {}

        # 1. Quadratic model (main specification)
        print(f"\n1. QUADRATIC MODEL:")
        quad_results = self._run_model_specification(
            analysis_df, y_var, var_sets['quadratic'], control_vars,
            'quadratic', sector_name, use_centered
        )
        if quad_results:
            all_results['quadratic'] = quad_results

        # 2. Cubic model
        print(f"\n2. CUBIC MODEL:")
        cubic_results = self._run_model_specification(
            analysis_df, y_var, var_sets['cubic'], control_vars,
            'cubic', sector_name, use_centered
        )
        if cubic_results:
            all_results['cubic'] = cubic_results

        # 3. Logarithmic model
        print(f"\n3. LOGARITHMIC MODEL:")
        log_results = self._run_model_specification(
            analysis_df, y_var, var_sets['logarithmic'], control_vars,
            'logarithmic', sector_name, use_centered
        )
        if log_results:
            all_results['logarithmic'] = log_results

        # 4. Square root model
        print(f"\n4. SQUARE ROOT MODEL:")
        sqrt_results = self._run_model_specification(
            analysis_df, y_var, var_sets['sqrt'], control_vars,
            'square_root', sector_name, use_centered
        )
        if sqrt_results:
            all_results['sqrt'] = sqrt_results

        # 5. Threshold models
        print(f"\n5. THRESHOLD MODELS:")
        threshold_results = self._run_threshold_models(
            analysis_df, y_var, control_vars, sector_name
        )
        if threshold_results:
            all_results['threshold'] = threshold_results

        # 6. Interaction model
        print(f"\n6. INTERACTION MODEL:")
        interaction_results = self._run_interaction_model(
            analysis_df, y_var, control_vars, sector_name, use_centered
        )
        if interaction_results:
            all_results['interaction'] = interaction_results

        # Model comparison
        print(f"\n7. MODEL COMPARISON:")
        comparison_df = self._compare_models(all_results)

        # Best model selection
        print(f"\n8. BEST MODEL SELECTION:")
        best_model_info = self._select_best_model(all_results)

        # Run diagnostics if requested
        if run_diagnostics and best_model_info:
            print(f"\n9. MODEL DIAGNOSTICS (Best Model):")
            self._run_complete_diagnostics(best_model_info, analysis_df)

        # Create summary table
        print(f"\n10. FINAL SUMMARY:")
        self._create_final_summary(all_results, best_model_info,
                                 comparison_df, sector_name, y_var, use_centered)

        return {
            'all_results': all_results,
            'best_model': best_model_info,
            'comparison': comparison_df,
            'sector': sector_name,
            'y_var': y_var,
            'approach': approach,
            'data': analysis_df
        }

    def _run_model_specification(self, df, y_var, esg_vars, control_vars,
                               model_type, sector_name, centered):
        """Run a single model specification"""
        # Prepare all variables
        all_vars = esg_vars + control_vars + [y_var]
        df_clean = df.dropna(subset=all_vars)

        if len(df_clean) < 10:
            print(f"  ⚠ Insufficient data: {len(df_clean)} observations")
            return None

        # Check VIF before running
        vif_info = self._calculate_vif(df_clean, esg_vars)

        print(f"  N = {len(df_clean)}, Variables = {len(esg_vars)}")
        print(f"  Max VIF = {vif_info['max_vif']:.2f}")

        if vif_info['max_vif'] > 10:
            print(f"  ⚠ High multicollinearity (VIF > 10)")

        # Run regression
        y = df_clean[y_var]
        X = df_clean[esg_vars + control_vars]
        X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

        try:
            model = PanelOLS(y, X, entity_effects=False, time_effects=False)
            results = model.fit(cov_type='robust')

            # Calculate turning points for quadratic/cubic models
            turning_points = None
            if 'quadratic' in model_type:
                turning_points = self._calculate_turning_points(
                    results, df_clean, esg_vars, centered
                )
            elif 'cubic' in model_type:
                turning_points = self._calculate_inflection_points(
                    results, df_clean, esg_vars, centered
                )

            # Store results
            result_info = {
                'model_type': model_type,
                'results': results,
                'esg_vars': esg_vars,
                'control_vars': control_vars,
                'nobs': results.nobs,
                'r2': results.rsquared,
                'r2_adj': results.rsquared_adj if hasattr(results, 'rsquared_adj') else results.rsquared,
                'vif_max': vif_info['max_vif'],
                'vif_details': vif_info,
                'turning_points': turning_points,
                'centered': centered,
                'sector': sector_name,
                'y_var': y_var,
                'data': df_clean
            }

            # Print detailed results
            self._print_model_details(result_info)

            # Store in global results
            key = f"{model_type}_{'centered' if centered else 'simple'}_{sector_name.replace(' ', '_')}_{y_var}"
            self.results[key] = result_info

            return result_info

        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            return None

    def _run_threshold_models(self, df, y_var, control_vars, sector_name):
        """Run threshold models with different cutoffs"""
        threshold_results = {}

        for component in ['E_lag', 'S_lag', 'G_lag']:
            for q in self.config.THRESHOLD_QUANTILES:
                threshold_var = f'{component}_above{q*100:.0f}'

                if threshold_var in df.columns:
                    # Prepare variables
                    esg_vars = [threshold_var]
                    all_vars = esg_vars + control_vars + [y_var]
                    df_clean = df.dropna(subset=all_vars)

                    if len(df_clean) < 10:
                        continue

                    # Run regression
                    y = df_clean[y_var]
                    X = df_clean[esg_vars + control_vars]
                    X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

                    try:
                        model = PanelOLS(y, X, entity_effects=False, time_effects=False)
                        results = model.fit(cov_type='robust')

                        # Store
                        key = f"threshold_{component}_q{q*100:.0f}"
                        threshold_results[key] = {
                            'model_type': 'threshold',
                            'results': results,
                            'component': component,
                            'quantile': q,
                            'threshold_var': threshold_var,
                            'nobs': results.nobs,
                            'r2': results.rsquared
                        }

                        # Print if significant
                        if threshold_var in results.params:
                            coeff = results.params[threshold_var]
                            pval = results.pvalues.get(threshold_var, 1.0)
                            if pval < 0.10:
                                threshold_val = df[component].quantile(q)
                                print(f"  ✓ {component} > {threshold_val:.2f}: β = {coeff:.4f} (p={pval:.3f})")

                    except:
                        pass

        return threshold_results

    def _run_interaction_model(self, df, y_var, control_vars, sector_name, centered):
        """Run interaction model"""
        # Define base variables
        if centered:
            base_vars = ['E_lag_centered', 'S_lag_centered', 'G_lag_centered']
            interactions = ['E_lag_centered_S_lag_centered_interact',
                           'E_lag_centered_G_lag_centered_interact',
                           'S_lag_centered_G_lag_centered_interact']
        else:
            base_vars = ['E_lag', 'S_lag', 'G_lag']
            # Create interactions if they don't exist
            interactions = []
            for i in range(len(base_vars)):
                for j in range(i+1, len(base_vars)):
                    var1, var2 = base_vars[i], base_vars[j]
                    inter_var = f'{var1}_{var2}_interact'
                    if inter_var not in df.columns:
                        df[inter_var] = df[var1] * df[var2]
                    interactions.append(inter_var)

        # Prepare all variables
        esg_vars = base_vars + interactions
        all_vars = esg_vars + control_vars + [y_var]
        df_clean = df.dropna(subset=all_vars)

        if len(df_clean) < 10:
            print(f"  ⚠ Insufficient data")
            return None

        # Run regression
        y = df_clean[y_var]
        X = df_clean[esg_vars + control_vars]
        X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

        try:
            model = PanelOLS(y, X, entity_effects=False, time_effects=False)
            results = model.fit(cov_type='robust')

            # Calculate marginal effects
            marginal_effects = {}
            for component in ['E', 'S', 'G']:
                base_var = f'{component}_lag_centered' if centered else f'{component}_lag'
                if base_var in results.params:
                    base_effect = results.params[base_var]
                    interaction_effect = 0

                    # Sum interaction effects
                    for other in ['E', 'S', 'G']:
                        if other != component:
                            other_var = f'{other}_lag_centered' if centered else f'{other}_lag'
                            inter_var = f'{base_var}_{other_var}_interact'
                            if inter_var in results.params:
                                other_mean = df_clean[other_var].mean()
                                interaction_effect += results.params[inter_var] * other_mean

                    marginal_effects[component] = {
                        'base_effect': base_effect,
                        'interaction_effect': interaction_effect,
                        'total_marginal_effect': base_effect + interaction_effect
                    }

            # Store results
            result_info = {
                'model_type': 'interaction',
                'results': results,
                'esg_vars': esg_vars,
                'control_vars': control_vars,
                'nobs': results.nobs,
                'r2': results.rsquared,
                'r2_adj': results.rsquared_adj if hasattr(results, 'rsquared_adj') else results.rsquared,
                'marginal_effects': marginal_effects,
                'centered': centered,
                'sector': sector_name,
                'y_var': y_var
            }

            # Print summary
            print(f"  N = {results.nobs}, R² = {results.rsquared:.6f}")
            if marginal_effects:
                for component, effects in marginal_effects.items():
                    print(f"  {component}: Marginal effect = {effects['total_marginal_effect']:.6f}")

            return result_info

        except Exception as e:
            print(f"  ✗ Error: {e}")
            return None

    def _calculate_vif(self, df, variables):
        """Calculate Variance Inflation Factors"""
        X_data = df[variables].dropna()

        if len(X_data) < 2:
            return {'max_vif': 0, 'details': {}}

        vif_details = {}
        for i, var in enumerate(variables):
            try:
                vif = variance_inflation_factor(X_data.values, i)
                vif_details[var] = vif
            except:
                vif_details[var] = np.nan

        valid_vifs = [v for v in vif_details.values() if not np.isnan(v)]
        max_vif = max(valid_vifs) if valid_vifs else 0

        return {
            'max_vif': max_vif,
            'details': vif_details,
            'high_vif_vars': [var for var, vif in vif_details.items()
                            if vif > 10 and not np.isnan(vif)]
        }

    def _calculate_turning_points(self, results, df, esg_vars, centered):
        """Calculate turning points for quadratic models"""
        turning_points = {}

        # Extract component from variable names
        for component in ['E', 'S', 'G']:
            # Find linear and quadratic variables for this component
            linear_var = None
            quad_var = None

            for var in esg_vars:
                if component in var:
                    if '_sq' in var or 'sq_' in var:
                        quad_var = var
                    elif 'cube' not in var:  # Exclude cubic
                        linear_var = var

            if linear_var and quad_var:
                if linear_var in results.params and quad_var in results.params:
                    beta1 = results.params[linear_var]
                    beta2 = results.params[quad_var]

                    if beta2 != 0:
                        turning_point_raw = -beta1 / (2 * beta2)

                        # Adjust for centering
                        if centered and '_centered' in linear_var:
                            # Get original variable name
                            orig_var = linear_var.replace('_centered', '')
                            if orig_var in df.columns:
                                mean_val = df[orig_var].mean()
                                turning_point = turning_point_raw + mean_val
                            else:
                                turning_point = turning_point_raw
                        else:
                            turning_point = turning_point_raw

                        # Check if within data range
                        if '_centered' in linear_var:
                            check_var = linear_var.replace('_centered', '')
                        else:
                            check_var = linear_var

                        if check_var in df.columns:
                            data_min = df[check_var].min()
                            data_max = df[check_var].max()
                            within_range = data_min <= turning_point <= data_max

                            shape = "U-shaped (convex)" if beta2 > 0 else "Inverted U-shaped (concave)"

                            turning_points[component] = {
                                'turning_point': turning_point,
                                'turning_point_raw': turning_point_raw,
                                'shape': shape,
                                'beta1': beta1,
                                'beta2': beta2,
                                'within_range': within_range,
                                'data_range': [data_min, data_max]
                            }

        return turning_points

    def _calculate_inflection_points(self, results, df, esg_vars, centered):
        """Calculate inflection points for cubic models"""
        inflection_points = {}

        for component in ['E', 'S', 'G']:
            # Find variables for this component
            linear_var = None
            quad_var = None
            cube_var = None

            for var in esg_vars:
                if component in var:
                    if 'cube' in var:
                        cube_var = var
                    elif '_sq' in var or 'sq_' in var:
                        quad_var = var
                    else:
                        linear_var = var

            if linear_var and quad_var and cube_var:
                if all(v in results.params for v in [linear_var, quad_var, cube_var]):
                    beta1 = results.params[linear_var]
                    beta2 = results.params[quad_var]
                    beta3 = results.params[cube_var]

                    if beta3 != 0:
                        # Inflection point: x = -beta2/(3*beta3)
                        inflection_point_raw = -beta2 / (3 * beta3)

                        # Adjust for centering
                        if centered and '_centered' in linear_var:
                            orig_var = linear_var.replace('_centered', '')
                            if orig_var in df.columns:
                                mean_val = df[orig_var].mean()
                                inflection_point = inflection_point_raw + mean_val
                            else:
                                inflection_point = inflection_point_raw
                        else:
                            inflection_point = inflection_point_raw

                        inflection_points[component] = {
                            'inflection_point': inflection_point,
                            'inflection_point_raw': inflection_point_raw,
                            'beta1': beta1,
                            'beta2': beta2,
                            'beta3': beta3
                        }

        return inflection_points

    def _print_model_details(self, result_info):
        """Print detailed model results"""
        results = result_info['results']

        print(f"  R² = {results.rsquared:.{self.config.R2_DECIMALS}f}, "
              f"Adj. R² = {result_info['r2_adj']:.{self.config.R2_DECIMALS}f}")

        # Print key coefficients
        print(f"  Key coefficients:")
        for var in result_info['esg_vars'][:3]:  # First 3 ESG variables
            if var in results.params:
                coeff = results.params[var]
                pval = results.pvalues.get(var, 1.0)
                sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                print(f"    {var}: {coeff:.{self.config.COEFF_DECIMALS}f}{sig}")

        # Print turning points if available
        if result_info.get('turning_points'):
            print(f"  Turning points:")
            for component, info in result_info['turning_points'].items():
                print(f"    {component}: {info['shape']} at {info['turning_point']:.4f}")

    def _compare_models(self, all_results):
        """Compare all model specifications"""
        if not all_results:
            return None

        comparison_data = []

        for model_type, result_info in all_results.items():
            if isinstance(result_info, dict) and 'results' in result_info:
                results = result_info['results']

                comparison_data.append({
                    'Model': result_info['model_type'].replace('_', ' ').title(),
                    'Approach': 'Centered' if result_info.get('centered', False) else 'Simple',
                    'N': result_info['nobs'],
                    'R²': result_info['r2'],
                    'Adj. R²': result_info['r2_adj'],
                    'Max VIF': result_info.get('vif_max', np.nan),
                    'AIC': self._calculate_aic(results),
                    'BIC': self._calculate_bic(results),
                    'ESG Vars': len(result_info['esg_vars'])
                })

        comparison_df = pd.DataFrame(comparison_data)

        if not comparison_df.empty:
            # Sort by R²
            comparison_df = comparison_df.sort_values('R²', ascending=False)

            print(f"\n{'Model':<20} {'Approach':<10} {'N':>6} {'R²':>12} {'Adj. R²':>12} {'Max VIF':>10} {'AIC':>12} {'BIC':>12}")
            print("-" * 110)

            for _, row in comparison_df.iterrows():
                print(f"{row['Model']:<20} {row['Approach']:<10} {row['N']:>6} "
                      f"{row['R²']:>12.{self.config.R2_DECIMALS}f} "
                      f"{row['Adj. R²']:>12.{self.config.R2_DECIMALS}f} "
                      f"{row['Max VIF']:>10.2f} "
                      f"{row['AIC']:>12.1f} {row['BIC']:>12.1f}")

        return comparison_df

    def _select_best_model(self, all_results):
        """Select best model using multiple criteria"""
        if not all_results:
            return None

        # Convert to list for easier processing
        model_list = []
        for model_type, result_info in all_results.items():
            if isinstance(result_info, dict) and 'results' in result_info:
                model_list.append({
                    'type': model_type,
                    'info': result_info,
                    'r2': result_info['r2'],
                    'bic': self._calculate_bic(result_info['results']),
                    'vif': result_info.get('vif_max', np.nan)
                })

        if not model_list:
            return None

        # Find best by different criteria
        best_by_r2 = max(model_list, key=lambda x: x['r2'])
        best_by_bic = min(model_list, key=lambda x: x['bic'])
        best_by_vif = min(model_list, key=lambda x: x['vif'])

        print(f"\nBest by R²: {best_by_r2['type']} (R² = {best_by_r2['r2']:.6f})")
        print(f"Best by BIC: {best_by_bic['type']} (BIC = {best_by_bic['bic']:.1f})")
        print(f"Best by VIF: {best_by_vif['type']} (VIF = {best_by_vif['vif']:.2f})")

        # Recommendation: Prioritize BIC (most conservative)
        recommendation = best_by_bic

        print(f"\nRECOMMENDED MODEL: {recommendation['type'].upper()}")
        print(f"  • R² = {recommendation['r2']:.6f}")
        print(f"  • BIC = {recommendation['bic']:.1f}")
        print(f"  • Max VIF = {recommendation['vif']:.2f}")
        print(f"  • Approach: {'Centered' if recommendation['info'].get('centered', False) else 'Simple'}")

        return recommendation['info']

    def _run_complete_diagnostics(self, model_info, df):
        """Run complete model diagnostics"""
        results = model_info['results']

        # 1. Multicollinearity (already calculated)
        print(f"\n1. Multicollinearity (VIF):")
        print(f"   Max VIF: {model_info.get('vif_max', 'N/A'):.2f}")
        if model_info.get('vif_details', {}).get('high_vif_vars'):
            print(f"   High VIF variables: {model_info['vif_details']['high_vif_vars']}")

        # 2. Functional form test (RESET)
        print(f"\n2. Functional Form (RESET test):")
        try:
            reset_result = linear_reset(results, power=2, test_type='fitted')
            print(f"   F-statistic: {reset_result.statistic:.4f}")
            print(f"   p-value: {reset_result.pvalue:.4f}")

            if reset_result.pvalue < 0.05:
                print(f"   ⚠ Reject linearity (supports non-linear specification)")
            else:
                print(f"   ✓ Cannot reject linearity")
        except Exception as e:
            print(f"   Could not perform RESET test: {str(e)[:50]}")

        # 3. Heteroskedasticity
        print(f"\n3. Heteroskedasticity (Breusch-Pagan):")
        try:
            bp_test = het_breuschpagan(results.resids, results.model.exog.dataframe)
            print(f"   LM statistic: {bp_test[0]:.4f}")
            print(f"   p-value: {bp_test[1]:.4f}")

            if bp_test[1] < 0.05:
                print(f"   ⚠ Evidence of heteroskedasticity (robust SE used)")
            else:
                print(f"   ✓ No evidence of heteroskedasticity")
        except:
            print(f"   Could not perform Breusch-Pagan test")

        # 4. Normality of residuals
        print(f"\n4. Normality of Residuals:")
        try:
            from scipy import stats
            residuals = results.resids

            # Jarque-Bera test
            jb_stat, jb_pval = stats.jarque_bera(residuals)
            print(f"   Jarque-Bera: {jb_stat:.4f} (p={jb_pval:.4f})")

            if jb_pval < 0.05:
                print(f"   ⚠ Reject normality of residuals")
            else:
                print(f"   ✓ Cannot reject normality")
        except:
            print(f"   Could not perform normality test")

    def _create_final_summary(self, all_results, best_model, comparison_df,
                            sector_name, y_var, use_centered):
        """Create final summary of analysis"""
        print("\n" + "="*80)
        print("FINAL ANALYSIS SUMMARY")
        print("="*80)

        print(f"\nAnalysis Details:")
        print(f"  • Sector: {sector_name}")
        print(f"  • Dependent variable: {y_var}")
        print(f"  • Approach: {'Centered (recommended)' if use_centered else 'Simple'}")
        print(f"  • Models tested: {len(all_results)}")

        if best_model:
            print(f"\nBest Model: {best_model['model_type'].upper()}")
            print(f"  • R²: {best_model['r2']:.{self.config.R2_DECIMALS}f}")
            print(f"  • Adjusted R²: {best_model['r2_adj']:.{self.config.R2_DECIMALS}f}")
            print(f"  • Observations: {best_model['nobs']}")
            print(f"  • Max VIF: {best_model.get('vif_max', 'N/A'):.2f}")

            # Print key findings
            if best_model['model_type'] == 'quadratic' and best_model.get('turning_points'):
                print(f"\nKey Findings (Quadratic Model):")
                for component, info in best_model['turning_points'].items():
                    print(f"  • {component}: {info['shape']}")
                    print(f"    Turning point: {info['turning_point']:.4f}")
                    print(f"    Within data range: {'Yes' if info['within_range'] else 'No'}")

            elif best_model['model_type'] == 'interaction' and best_model.get('marginal_effects'):
                print(f"\nKey Findings (Interaction Model):")
                for component, effects in best_model['marginal_effects'].items():
                    print(f"  • {component}: Total marginal effect = {effects['total_marginal_effect']:.6f}")

        # Save results if requested
        if self.config.SAVE_RESULTS:
            self._save_all_results(all_results, comparison_df, sector_name, y_var)

    def _save_all_results(self, all_results, comparison_df, sector_name, y_var):
        """Save all results to Excel"""
        os.makedirs(self.config.OUTPUT_DIR, exist_ok=True)

        # Create filename
        filename = f"nonlinear_results_{sector_name.replace(' ', '_')}_{y_var}.xlsx"
        filepath = os.path.join(self.config.OUTPUT_DIR, filename)

        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            # 1. Model comparison
            if comparison_df is not None:
                comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)

            # 2. Detailed results for each model
            for i, (model_type, result_info) in enumerate(all_results.items()):
                if isinstance(result_info, dict) and 'results' in result_info:
                    # Create coefficient table
                    results = result_info['results']

                    coeff_df = pd.DataFrame({
                        'Variable': results.params.index,
                        'Coefficient': results.params.values,
                        'Std_Error': [results.std_errors.get(v, np.nan) for v in results.params.index],
                        't_Stat': [results.tstats.get(v, np.nan) for v in results.params.index],
                        'P_Value': [results.pvalues.get(v, np.nan) for v in results.params.index]
                    })

                    # Limit sheet name length
                    sheet_name = f"{model_type}_{i}"[:31]
                    coeff_df.to_excel(writer, sheet_name=sheet_name, index=False)

            # 3. Summary statistics
            summary_data = []
            for model_type, result_info in all_results.items():
                if isinstance(result_info, dict) and 'results' in result_info:
                    summary_data.append({
                        'Model': model_type,
                        'Approach': 'Centered' if result_info.get('centered', False) else 'Simple',
                        'N': result_info['nobs'],
                        'R2': result_info['r2'],
                        'Adj_R2': result_info['r2_adj'],
                        'Max_VIF': result_info.get('vif_max', np.nan)
                    })

            if summary_data:
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='Summary', index=False)

        print(f"\n✓ All results saved to: {filepath}")
        return filepath

    def _calculate_aic(self, results):
        """Calculate Akaike Information Criterion"""
        n = results.nobs
        k = len(results.params)
        ssr = getattr(results, 'ssr', None)

        if ssr is None:
            # Estimate SSR from residuals
            y_pred = results.predict().fitted_values
            y_actual = results.model.dependent.dataframe.iloc[:, 0]
            ssr = np.sum((y_actual - y_pred) ** 2)

        return n * np.log(ssr/n) + 2 * k

    def _calculate_bic(self, results):
        """Calculate Bayesian Information Criterion"""
        n = results.nobs
        k = len(results.params)
        ssr = getattr(results, 'ssr', None)

        if ssr is None:
            # Estimate SSR from residuals
            y_pred = results.predict().fitted_values
            y_actual = results.model.dependent.dataframe.iloc[:, 0]
            ssr = np.sum((y_actual - y_pred) ** 2)

        return n * np.log(ssr/n) + k * np.log(n)

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution function"""
    print("\n" + "="*80)
    print("COMPLETE NON-LINEAR EFFECTS ANALYSIS")
    print("With Centered and Simple Approaches")
    print("="*80)

    # Initialize configuration
    config = Config()

    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    # Initialize analyzer
    analyzer = NonlinearEffectsAnalyzerComplete(config)

    # Run analysis for Consumer Staples (recommended for non-linear)
    print("\n" + "="*80)
    print("ANALYSIS 1: CONSUMER STAPLES (Recommended for non-linear effects)")
    print("="*80)

    results_cs = analyzer.run_complete_analysis(
        sector='cs',
        y_var='Tobin_Q',
        use_centered=config.USE_CENTERED_APPROACH,
        run_diagnostics=True
    )

    # Run analysis for All Sectors (for comparison)
    print("\n" + "="*80)
    print("ANALYSIS 2: ALL SECTORS (Comparison)")
    print("="*80)

    results_all = analyzer.run_complete_analysis(
        sector='all',
        y_var='Tobin_Q',
        use_centered=config.USE_CENTERED_APPROACH,
        run_diagnostics=False  # Skip diagnostics for comparison
    )

    # Run with log-transformed Tobin's Q (robustness)
    print("\n" + "="*80)
    print("ANALYSIS 3: LOG-TRANSFORMED TOBIN'S Q (Robustness)")
    print("="*80)

    results_log = analyzer.run_complete_analysis(
        sector='cs',
        y_var='Tobin_Q_log',
        use_centered=config.USE_CENTERED_APPROACH,
        run_diagnostics=False
    )

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)

    print(f"\nSummary:")
    print(f"  • Output directory: {config.OUTPUT_DIR}")
    print(f"  • Approach used: {'CENTERED (recommended)' if config.USE_CENTERED_APPROACH else 'SIMPLE'}")
    print(f"  • Models run: {len(analyzer.results)}")
    print(f"  • Files saved: Check {config.OUTPUT_DIR}")

    return analyzer, results_cs, results_all, results_log

# ============================================================================
# EXECUTION
# ============================================================================
if __name__ == "__main__":
    analyzer, results_cs, results_all, results_log = main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

COMPLETE NON-LINEAR EFFECTS ANALYSIS
With Centered and Simple Approaches

ANALYSIS 1: CONSUMER STAPLES (Recommended for non-linear effects)

COMPLETE NON-LINEAR ANALYSIS - CENTERED APPROACH
Sector: Consumer Staples
Dependent: Tobin_Q
Loading data and creating all non-linear transformations...
------------------------------------------------------------
Creating lagged ESG variables...
Creating sector variable...

Creating non-linear transformations...

A. CENTERED TRANSFORMATIONS (Recommended):
----------------------------------------
  ✓ E_lag_centered = E_lag - 72.3449
  ✓ E_lag_sq_centered = (E_lag_centered)²
  ✓ E_lag_cube_centered = (E_lag_centered)³
  ✓ S_lag_centered = S_lag - 74.8235
  ✓ S_lag_sq_centered = (S_lag_centered)²
  ✓ S_lag_cube_centered = (S_lag_centered)³
  ✓ G_lag_centered = G_lag - 73.6079
  ✓ G_lag_sq_centered = (G_lag_centered)²
  ✓ 